# 🍳 Optimizer Cookbook — Chapter 04: Adam

**Previous:** `03_rmsprop.ipynb` — RMSprop  
**Next:** `05_adamw.ipynb` — AdamW

---

## The Optimizer That Changed Everything

By 2015, practitioners had two good ideas but neither was complete:

- **SGD + Momentum** — accelerates gradients in consistent directions, but uses a single global LR
- **RMSprop** — adapts LR per parameter using gradient EMA, but has no momentum

**Adam** (Adaptive Moment Estimation, Kingma & Ba, 2015) unifies both into a single optimizer — and became the default choice for virtually all deep learning within a year of publication.

---

## Core Idea: Two Moment Estimates

Adam maintains two running estimates per parameter:

- **First moment** $m_t$ — EMA of gradients (like momentum)
- **Second moment** $v_t$ — EMA of squared gradients (like RMSprop)

---

## Update Rule

**Step 1 — Compute moment estimates:**
$$m_t = \beta_1 \cdot m_{t-1} + (1 - \beta_1) \cdot \nabla_{\theta} \mathcal{L}$$
$$v_t = \beta_2 \cdot v_{t-1} + (1 - \beta_2) \cdot (\nabla_{\theta} \mathcal{L})^2$$

**Step 2 — Bias correction** (critical at early timesteps when $m_t, v_t \approx 0$):
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

**Step 3 — Parameter update:**
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

Where:
- $\beta_1$ = first moment decay (default **0.9**) — momentum-like smoothing
- $\beta_2$ = second moment decay (default **0.999**) — RMSprop-like LR scaling
- $\epsilon$ = numerical stability (default **1e-8**)
- $\eta$ = learning rate (default **0.001**)

---

## Why Bias Correction Matters

At $t=1$: $m_1 = (1-\beta_1) \cdot g_1$ — this is close to **zero**, not the actual gradient.  
Without bias correction, Adam takes tiny steps at the start. The bias-corrected $\hat{m}_t$ rescales this properly.

This is what separates Adam from a naive combination of Momentum + RMSprop.

---

## The Weight Decay Problem (Why AdamW Exists)

Standard Adam applies L2 regularization by adding $\lambda \theta$ to the gradient **before** computing moments:  
$$\nabla' = \nabla \mathcal{L} + \lambda \theta$$

This **couples** weight decay with the adaptive LR — the regularization effect is scaled down by $\frac{1}{\sqrt{\hat{v}_t}}$, making it weaker for frequently-updated parameters. This is a known flaw. **AdamW** (next chapter) fixes it.

---

## When to Use Adam

| Scenario | Verdict |
|---|---|
| Default choice for any new task | ✅ Excellent |
| CNNs, MLPs, autoencoders | ✅ Excellent |
| NLP — RNNs, LSTMs, GRUs | ✅ Excellent |
| GANs | ✅ Very commonly used |
| Quick prototyping | ✅ Best default — works without tuning |
| Transformers / BERT / GPT | ⚠️ Use AdamW instead (proper weight decay) |
| Fine-tuning pretrained models | ⚠️ Use AdamW instead |
| Very large batch training | ⚠️ LAMB/LARS preferred |

---

## Key Hyperparameters

| Parameter | Default | Notes |
|---|---|---|
| `lr` | 1e-3 | Rarely needs changing — Adam is robust |
| `betas` | (0.9, 0.999) | β₁ controls momentum, β₂ controls LR scaling |
| `eps` | 1e-8 | Increase to 1e-7 or 1e-6 if training is unstable |
| `weight_decay` | 0 | ⚠️ Coupled with adaptive LR — use AdamW for proper L2 |

---
## 0. Imports & CUDA Check

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import os, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')

---
## 1. Config

In [ ]:
BATCH_SIZE     = 128
NUM_EPOCHS     = 20
NUM_CLASSES    = 10
NUM_WORKERS    = 2
SEED           = 42

# ── Optimizer Config ─────────────────────────────────────────
LEARNING_RATE  = 1e-3
BETA1          = 0.9       # first moment decay
BETA2          = 0.999     # second moment decay
EPS            = 1e-8
WEIGHT_DECAY   = 0         # ⚠️ intentionally 0 — see theory above
OPTIMIZER_NAME = 'Adam'

DATA_DIR    = '../data'
RESULTS_DIR = '../results/logs'
PLOTS_DIR   = '../results/plots'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Config loaded ✓')

---
## 2. Dataset — CIFAR-10

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

---
## 3. Model — SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(SEED)
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready. Trainable params: {total_params:,}')

---
## 4. Optimizer — Adam

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr           = LEARNING_RATE,
    betas        = (BETA1, BETA2),
    eps          = EPS,
    weight_decay = WEIGHT_DECAY
)

print(f'Optimizer    : Adam')
print(f'LR           : {LEARNING_RATE}')
print(f'Betas        : β₁={BETA1}, β₂={BETA2}')
print(f'Eps          : {EPS}')
print(f'Weight Decay : {WEIGHT_DECAY}  (see AdamW chapter for proper L2)')
print('-' * 65)

---
## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total

def run_training(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, optimizer_name):
    history = []
    best_acc = 0.0
    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc     = evaluate(model, test_loader, criterion, device)
        elapsed = time.time() - t0
        if val_acc > best_acc: best_acc = val_acc
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_loss, 'val_acc': val_acc, 'time_s': elapsed})
        print(f'Epoch [{epoch:02d}/{num_epochs}] Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | Time: {elapsed:.1f}s')
    print(f'\n✓ Best Val Accuracy: {best_acc:.2f}%')
    df = pd.DataFrame(history)
    df.to_csv(f'{RESULTS_DIR}/{optimizer_name}_log.csv', index=False)
    print(f'✓ Log saved → results/logs/{optimizer_name}_log.csv')
    return df

print('Utilities loaded ✓')

---
## 6. Train

In [ ]:
history = run_training(
    model          = model,
    train_loader   = train_loader,
    test_loader    = test_loader,
    optimizer      = optimizer,
    criterion      = criterion,
    num_epochs     = NUM_EPOCHS,
    device         = device,
    optimizer_name = OPTIMIZER_NAME
)

---
## 7. Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — {OPTIMIZER_NAME}', fontsize=14, fontweight='bold')

ax1.plot(history['epoch'], history['train_loss'], label='Train Loss', linewidth=2, color='steelblue')
ax1.plot(history['epoch'], history['val_loss'],   label='Val Loss',   linewidth=2, color='tomato', linestyle='--')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['epoch'], history['train_acc'], label='Train Acc', linewidth=2, color='steelblue')
ax2.plot(history['epoch'], history['val_acc'],   label='Val Acc',   linewidth=2, color='tomato', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/{OPTIMIZER_NAME}_curves.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Bias Correction Visualisation

Demonstrates why bias correction matters in the early steps of training.  
We compare the **raw** first moment $m_t$ vs the **bias-corrected** $\hat{m}_t$ for β₁=0.9.

In [ ]:
# Simulate bias correction effect analytically
beta1 = 0.9
steps = 50
g = 1.0  # assume constant gradient = 1 for illustration

m_raw   = []
m_corrected = []
m = 0.0

for t in range(1, steps + 1):
    m = beta1 * m + (1 - beta1) * g
    m_hat = m / (1 - beta1 ** t)
    m_raw.append(m)
    m_corrected.append(m_hat)

plt.figure(figsize=(10, 4))
plt.plot(range(1, steps+1), m_raw,       label='Raw $m_t$ (no bias correction)', linewidth=2, color='tomato',     linestyle='--')
plt.plot(range(1, steps+1), m_corrected, label='Bias-corrected $\\hat{m}_t$',    linewidth=2, color='steelblue')
plt.axhline(y=1.0, color='gray', linestyle=':', linewidth=1.5, label='True gradient (g=1)')
plt.title('Adam — Bias Correction Effect on First Moment (β₁=0.9)', fontsize=12, fontweight='bold')
plt.xlabel('Training Step'); plt.ylabel('Moment Value')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/Adam_bias_correction.png', dpi=120, bbox_inches='tight')
plt.show()
print('Observation: Without bias correction, Adam takes much smaller steps early in training.')
print(f'At step 1 — Raw: {m_raw[0]:.4f} | Corrected: {m_corrected[0]:.4f} | True: 1.0')

---
## 9. LR Sensitivity Sweep

Adam is robust to LR, but let's verify empirically across a wide range.

In [ ]:
lr_values    = [1e-4, 1e-3, 5e-3, 1e-2]
sweep_results = {}
SWEEP_EPOCHS  = 5

for lr in lr_values:
    torch.manual_seed(SEED)
    m   = SimpleCNN(num_classes=NUM_CLASSES).to(device)
    opt = optim.Adam(m.parameters(), lr=lr, betas=(BETA1, BETA2), eps=EPS)
    crit = nn.CrossEntropyLoss()
    val_accs = []
    for epoch in range(1, SWEEP_EPOCHS + 1):
        train_one_epoch(m, train_loader, crit, opt, device)
        _, val_acc = evaluate(m, test_loader, crit, device)
        val_accs.append(val_acc)
    sweep_results[f'lr={lr}'] = val_accs
    print(f'lr={lr:.0e} → Final Val Acc: {val_accs[-1]:.2f}%')

plt.figure(figsize=(9, 5))
for label, accs in sweep_results.items():
    plt.plot(range(1, SWEEP_EPOCHS + 1), accs, marker='o', linewidth=2, label=label)
plt.title('Adam — LR Sensitivity (β₁=0.9, β₂=0.999)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy (%)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/Adam_lr_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Gradient Norm Tracking

Tracks the gradient norm per epoch — useful for spotting exploding/vanishing gradients.  
Adam's adaptive scaling should keep this relatively stable.

In [ ]:
torch.manual_seed(SEED)
track_model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
track_opt   = optim.Adam(track_model.parameters(), lr=LEARNING_RATE, betas=(BETA1, BETA2), eps=EPS)
track_crit  = nn.CrossEntropyLoss()

grad_norms = []
TRACK_EPOCHS = NUM_EPOCHS

for epoch in range(1, TRACK_EPOCHS + 1):
    track_model.train()
    epoch_norms = []
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        track_opt.zero_grad()
        loss = track_crit(track_model(images), labels)
        loss.backward()
        # compute total gradient norm before stepping
        total_norm = torch.nn.utils.clip_grad_norm_(track_model.parameters(), max_norm=float('inf'))
        epoch_norms.append(total_norm.item())
        track_opt.step()
    grad_norms.append(np.mean(epoch_norms))

plt.figure(figsize=(9, 4))
plt.plot(range(1, TRACK_EPOCHS+1), grad_norms, color='mediumpurple', linewidth=2, marker='o', markersize=4)
plt.title('Adam — Mean Gradient Norm per Epoch', fontsize=12, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Gradient Norm')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/Adam_gradient_norms.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Start norm: {grad_norms[0]:.4f} | End norm: {grad_norms[-1]:.4f}')

---
## 11. Full Comparison — All Optimizers So Far

In [ ]:
logs = {
    'Vanilla SGD'    : ('SGD_baseline_log.csv',  'gray',           ':'),
    'SGD + Momentum' : ('SGD_Momentum_log.csv',  'steelblue',      '--'),
    'Adagrad'        : ('Adagrad_log.csv',        'darkorange',     '--'),
    'RMSprop'        : ('RMSprop_log.csv',        'mediumseagreen', '--'),
    'Adam'           : ('Adam_log.csv',           'crimson',        '-'),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('All Optimizers — Cumulative Comparison', fontsize=14, fontweight='bold')

for label, (fname, color, style) in logs.items():
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path)
        lw = 2.5 if label == 'Adam' else 1.5
        ax1.plot(df['epoch'], df['val_loss'], label=label, color=color, linestyle=style, linewidth=lw)
        ax2.plot(df['epoch'], df['val_acc'],  label=label, color=color, linestyle=style, linewidth=lw)
    else:
        print(f'⚠️  Missing: {fname}')

ax1.set_title('Validation Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/comparison_up_to_adam.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 12. Leaderboard So Far

In [ ]:
logs_order = [
    ('Vanilla SGD',    'SGD_baseline_log.csv'),
    ('SGD + Momentum', 'SGD_Momentum_log.csv'),
    ('Adagrad',        'Adagrad_log.csv'),
    ('RMSprop',        'RMSprop_log.csv'),
    ('Adam',           'Adam_log.csv'),
]

print(f'{'Optimizer':<20} {'Best Val Acc':>14} {'Best Epoch':>11} {'Avg Time/Epoch':>16}')
print('-' * 65)
for label, fname in logs_order:
    path = f'{RESULTS_DIR}/{fname}'
    if os.path.exists(path):
        df  = pd.read_csv(path)
        best = df.loc[df['val_acc'].idxmax()]
        print(f"{label:<20} {best['val_acc']:>13.2f}% {int(best['epoch']):>11} {df['time_s'].mean():>14.1f}s")
    else:
        print(f'{label:<20} {"(not run yet)":>14}')
print('-' * 65)

---
## 13. Results Summary & Analysis

In [ ]:
best_epoch = history.loc[history['val_acc'].idxmax()]

print('=' * 55)
print(f'  {OPTIMIZER_NAME} — Summary')
print('=' * 55)
print(f"  Best Val Accuracy : {best_epoch['val_acc']:.2f}%")
print(f"  Best Val Loss     : {best_epoch['val_loss']:.4f}")
print(f"  Achieved at Epoch : {int(best_epoch['epoch'])}")
print(f"  Avg Time/Epoch    : {history['time_s'].mean():.1f}s")
print('=' * 55)
print()
print('📌 Key Observations:')
print('  ✅ Fastest convergence of all optimizers so far')
print('  ✅ Works well out-of-the-box with default hyperparameters')
print('  ✅ Bias correction gives stable early training')
print('  ✅ Gradient norm stays controlled — no explosion or vanishing')
print('  ⚠️  Can overfit faster than SGD+Momentum on small datasets')
print('  ⚠️  Weight decay is coupled with adaptive LR — use AdamW instead')
print('  ⚠️  Sometimes finds a slightly worse final minimum than SGD (generalisation gap)')
print()
print('📌 When to use Adam:')
print('  → Your first optimizer choice for any new task')
print('  → NLP (RNNs, LSTMs, attention models)')
print('  → GANs (discriminator and generator both benefit)')
print('  → When you cannot afford extensive LR tuning')

---
## 14. What's Next?

Adam is excellent, but it has one well-known flaw: **weight decay is not applied cleanly**.  
Because L2 regularization is folded into the gradient before adaptive scaling, parameters that update frequently receive *less* regularization than intended.

**AdamW** (Loshchilov & Hutter, 2019) decouples weight decay from the gradient update entirely — and is now the **recommended default for Transformers, BERT, GPT, and any fine-tuning task**.

➡️ Continue to `05_adamw.ipynb`